# IndicHallBench — Complete Human Validation and Statistical Analysis

**Study design:** two independent annotators; 3,600 response records; six languages; six models (verified from data). The sampling unit is a question within a language: select 100 questions per language and retain all six model responses for each selected question.

**Important treatment of the existing 1,015 human-reviewed records:** keep them as a separate targeted review set. They were selected from the pending/error pool and were reviewed previously; they are not a representative random sample and do not independently establish inter-annotator agreement.

Run cells sequentially. The notebook first combines the annotation CSV, original responses JSON, and references JSON. It does not assume the annotation CSV contains response text.

In [7]:
# 1. Imports, paths and study configuration
from pathlib import Path
import json, re, ast, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    cohen_kappa_score, classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support, f1_score
)
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

SEED = 42
PROJECT = Path(r"C:/Users/CH.SC.U4CSE23160/Neural Project/IndicHallbench-Hallucinations-Benchmark")

# Confirm these paths match your actual project folders.
ANNOTATION_PATH = PROJECT / "data/annotations/final/final_annotations.csv"
RESPONSE_PATH = PROJECT / "data/annotations/working/responses.json"
REFERENCE_PATH = PROJECT / "data/annotations/working/references.json"

VAL = PROJECT / "data/validation"
ANN = VAL / "annotator_files"
RES = PROJECT / "results/validation"
for folder in [VAL, ANN, RES]:
    folder.mkdir(parents=True, exist_ok=True)

LANGUAGES = ["EN", "HI", "TA", "TE", "NE", "HI-EN"]
QUESTIONS_PER_LANGUAGE = 100
EXPECTED_MODELS = 6
EXPECTED_RESPONSES = len(LANGUAGES) * QUESTIONS_PER_LANGUAGE * EXPECTED_MODELS

print("Expected validation response rows:", EXPECTED_RESPONSES)
print("Annotation file:", ANNOTATION_PATH)
print("Responses file:", RESPONSE_PATH)
print("References file:", REFERENCE_PATH)

Expected validation response rows: 3600
Annotation file: C:\Users\CH.SC.U4CSE23160\Neural Project\IndicHallbench-Hallucinations-Benchmark\data\annotations\final\final_annotations.csv
Responses file: C:\Users\CH.SC.U4CSE23160\Neural Project\IndicHallbench-Hallucinations-Benchmark\data\annotations\working\responses.json
References file: C:\Users\CH.SC.U4CSE23160\Neural Project\IndicHallbench-Hallucinations-Benchmark\data\annotations\working\references.json


## 2. Load and inspect the three source files

The annotation file contains labels and `response_id`, but it may not contain the original response text. The response file supplies the model response. The reference file supplies the question and ground truth.

The merge keys are:
- annotations ↔ responses: `question_id + language + model_name`
- enriched records ↔ references: `question_id`

The reference join intentionally uses `question_id`, not `fact_id`, because reference metadata may be sparse in `fact_id`.

In [8]:
# 2. Load annotations, responses and references
for path in [ANNOTATION_PATH, RESPONSE_PATH, REFERENCE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

annotations = pd.read_csv(ANNOTATION_PATH, encoding="utf-8-sig", low_memory=False)
responses = pd.read_json(RESPONSE_PATH)
references = pd.read_json(REFERENCE_PATH)

for name, frame in [
    ("annotations", annotations),
    ("responses", responses),
    ("references", references)
]:
    frame.columns = [str(c).strip() for c in frame.columns]
    print(f"{name}: {frame.shape}")
    print("Columns:", frame.columns.tolist())

annotations: (64800, 20)
Columns: ['response_id', 'question_id', 'fact_id', 'language', 'model_name', 'timestamp', 'judge_model', 'annotation_timestamp', 'correctness_label', 'hallucination_types', 'severity', 'claims', 'reason', 'confidence', 'review_required', 'judge_prompt_tokens', 'judge_output_tokens', 'annotation_source', 'annotator_model', 'human_annotator']
responses: (64800, 10)
Columns: ['question_id', 'fact_id', 'domain', 'difficulty', 'fact_type', 'language', 'model_name', 'response', 'timestamp', 'source_url']
references: (10800, 11)
Columns: ['question_id', 'fact_id', 'domain', 'difficulty', 'question_type', 'question', 'ground_truth', 'answer_type', 'answer_length_category', 'primary_hallucination_type', 'source_url']


In [9]:
# 3. Helper functions and column validation
def find_column(frame, candidates, required=True):
    lookup = {
        re.sub(r"[^a-z0-9]", "", str(c).lower()): c
        for c in frame.columns
    }
    for candidate in candidates:
        key = re.sub(r"[^a-z0-9]", "", candidate.lower())
        if key in lookup:
            return lookup[key]
    if required:
        raise KeyError(
            f"Could not find {candidates}. Available columns: {frame.columns.tolist()}"
        )
    return None

def clean_key(series):
    return series.astype("string").str.strip()

def normalize_language(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA
    value = str(value).strip().upper().replace("_", "-").replace(" ", "")
    mapping = {
        "EN": "EN", "ENGLISH": "EN",
        "HI": "HI", "HINDI": "HI",
        "TA": "TA", "TAMIL": "TA",
        "TE": "TE", "TELUGU": "TE",
        "NE": "NE", "NEPALI": "NE",
        "HIEN": "HI-EN", "HI-EN": "HI-EN"
    }
    return mapping.get(value, value)

# Required source columns
A_QID = find_column(annotations, ["question_id"])
A_LANG = find_column(annotations, ["language"])
A_MODEL = find_column(annotations, ["model_name", "model"])
A_RID = find_column(annotations, ["response_id"])

R_QID = find_column(responses, ["question_id"])
R_LANG = find_column(responses, ["language"])
R_MODEL = find_column(responses, ["model_name", "model"])
R_TEXT = find_column(responses, ["response", "model_response", "answer"])

REF_QID = find_column(references, ["question_id"])
REF_QUESTION = find_column(references, ["question", "prompt"])
REF_GOLD = find_column(references, ["ground_truth", "reference_answer", "gold_answer"])

print("Annotation response ID:", A_RID)
print("Response text column:", R_TEXT)
print("Reference question:", REF_QUESTION)
print("Reference answer:", REF_GOLD)

Annotation response ID: response_id
Response text column: response
Reference question: question
Reference answer: ground_truth


In [10]:
# 4. Normalize join keys and check source integrity
for frame, qid_col, lang_col, model_col in [
    (annotations, A_QID, A_LANG, A_MODEL),
    (responses, R_QID, R_LANG, R_MODEL)
]:
    frame[qid_col] = clean_key(frame[qid_col])
    frame[lang_col] = frame[lang_col].apply(normalize_language).astype("string")
    frame[model_col] = clean_key(frame[model_col])

references[REF_QID] = clean_key(references[REF_QID])

# Standardize key column names so the merge is explicit.
annotations = annotations.rename(columns={
    A_QID: "question_id", A_LANG: "language", A_MODEL: "model_name",
    A_RID: "response_id"
})
responses = responses.rename(columns={
    R_QID: "question_id", R_LANG: "language", R_MODEL: "model_name",
    R_TEXT: "response"
})
references = references.rename(columns={
    REF_QID: "question_id", REF_QUESTION: "question", REF_GOLD: "ground_truth"
})

if annotations[["question_id","language","model_name","response_id"]].isna().any().any():
    raise ValueError("Annotations have missing join keys or response_id.")
if responses[["question_id","language","model_name","response"]].isna().any().any():
    print("Missing response text/key counts:")
    display(responses[["question_id","language","model_name","response"]].isna().sum())
    raise ValueError("Responses have missing join keys or response text.")
if references[["question_id","question","ground_truth"]].isna().any().any():
    print("Missing reference fields:")
    display(references[["question_id","question","ground_truth"]].isna().sum())
    raise ValueError("References have missing question IDs, questions or ground truths.")

print("Annotation language counts:")
display(annotations["language"].value_counts(dropna=False))
print("Response language counts:")
display(responses["language"].value_counts(dropna=False))

Annotation language counts:


language
EN       10800
HI       10800
HI-EN    10800
NE       10800
TA       10800
TE       10800
Name: count, dtype: Int64

Response language counts:


language
EN       10800
HI       10800
HI-EN    10800
NE       10800
TA       10800
TE       10800
Name: count, dtype: Int64

In [12]:
print("Language distribution:")
display(
    responses["language"]
    .value_counts(dropna=False)
)

print("\nMissing language question IDs:")
print(missing_language["question_id"].tolist())

Language distribution:


language
EN       10800
HI       10800
HI-EN    10800
NE       10800
TA       10800
TE       10800
Name: count, dtype: Int64


Missing language question IDs:
[]


In [13]:
required_columns = [
    "question_id",
    "language",
    "model_name",
    "response"
]

missing_counts = responses[required_columns].isna().sum()

print("Missing response text/key counts:")
display(missing_counts)

if missing_counts.any():
    raise ValueError(
        "Responses still contain missing join keys or response text."
    )

print("Validation passed.")

Missing response text/key counts:


question_id    0
language       0
model_name     0
response       0
dtype: int64

Validation passed.


In [14]:
# 5. Merge annotations with responses, then with references
JOIN_KEYS = ["question_id", "language", "model_name"]

# A response must be unique per question-language-model.
dup_resp = responses.duplicated(JOIN_KEYS, keep=False)
if dup_resp.any():
    display(responses.loc[dup_resp, JOIN_KEYS].sort_values(JOIN_KEYS).head(30))
    raise ValueError("Duplicate response question-language-model keys. Resolve before merging.")

dup_ann = annotations.duplicated(JOIN_KEYS, keep=False)
if dup_ann.any():
    display(annotations.loc[dup_ann, JOIN_KEYS + ["response_id"]].head(30))
    raise ValueError("Duplicate annotation question-language-model keys. Resolve before merging.")

# Response ID should uniquely identify an annotation row.
if annotations["response_id"].duplicated().any():
    raise ValueError("response_id is duplicated in annotations.")

# Avoid accidental duplicate columns: retain response payload and response metadata;
# annotation labels remain from annotations.
response_payload_cols = [
    c for c in responses.columns
    if c not in JOIN_KEYS
]
enriched = annotations.merge(
    responses[JOIN_KEYS + response_payload_cols],
    on=JOIN_KEYS,
    how="left",
    validate="one_to_one",
    indicator="_response_match"
)

print("Annotation-to-response merge:")
display(enriched["_response_match"].value_counts())

unmatched_response = enriched[enriched["_response_match"] != "both"]
if not unmatched_response.empty:
    display(unmatched_response[JOIN_KEYS + ["response_id"]].head(30))
    raise ValueError("Some annotation rows did not match original responses.")

enriched = enriched.drop(columns="_response_match")

# References must be unique per question_id for a many-to-one join.
if references["question_id"].duplicated().any():
    dups = references[references["question_id"].duplicated(keep=False)]
    display(dups[["question_id","question","ground_truth"]].head(30))
    raise ValueError("references.json has duplicate question_id values.")

# Keep only one copy of the reference columns needed for analysis.
reference_payload_cols = [
    c for c in references.columns if c != "question_id"
]
df = enriched.merge(
    references[["question_id"] + reference_payload_cols],
    on="question_id",
    how="left",
    validate="many_to_one",
    indicator="_reference_match",
    suffixes=("", "_reference")
)

print("Response-to-reference merge:")
display(df["_reference_match"].value_counts())

unmatched_reference = df[df["_reference_match"] != "both"]
if not unmatched_reference.empty:
    display(unmatched_reference[["response_id","question_id"]].head(30))
    raise ValueError("Some responses have no matching reference question.")

df = df.drop(columns="_reference_match")

# Protect against response_id loss or accidental row multiplication.
assert len(df) == len(annotations), "Merge changed annotation row count."
assert df["response_id"].is_unique, "Merged response_id is not unique."
assert df["response"].notna().all()
assert df["question"].notna().all()
assert df["ground_truth"].notna().all()

print("Enriched dataset shape:", df.shape)
display(df.head(3))

Annotation-to-response merge:


_response_match
both          64800
left_only         0
right_only        0
Name: count, dtype: int64

Response-to-reference merge:


_reference_match
both          64800
left_only         0
right_only        0
Name: count, dtype: int64

Enriched dataset shape: (64800, 37)


,response_id,question_id,fact_id_x,language,model_name,timestamp_x,judge_model,annotation_timestamp,correctness_label,hallucination_types,...,fact_id,domain_reference,difficulty_reference,question_type,question,ground_truth,answer_type,answer_length_category,primary_hallucination_type,source_url_reference
0,7dcc63c133de839d15aa,Q_EN_CUL_001,CUL_001,EN,GPT-5.6 Luna,2026-10-07T13:18:23+05:30,llama3.1:8b,2026-11-03T07:58:00.027844+00:00,Correct,"[""None""]",...,CUL_001,Culture,Low,Entity,"Who coined the term Kulturbrille, or 'culture ...",Franz Boas,Short,1-5 words,Entity,https://en.wikipedia.org/wiki/Culture
1,bcfa10ff9c06932f320a,Q_EN_CUL_002,CUL_002,EN,GPT-5.6 Luna,2026-10-07T13:18:23+05:30,llama3.1:8b,2026-11-03T07:58:04.471048+00:00,Correct,"[""None""]",...,CUL_002,Culture,Low,Entity,Which ancient Roman orator used the term on wh...,Cicero,Short,1-5 words,Entity,https://en.wikipedia.org/wiki/Culture
2,e3baa0866d5520e618bf,Q_EN_CUL_003,CUL_003,EN,GPT-5.6 Luna,2026-10-07T13:18:23+05:30,llama3.1:8b,2026-11-03T07:58:08.993632+00:00,Correct,"[""None""]",...,CUL_003,Culture,Low,Entity,Which organization administers the internation...,UNESCO,Short,1-5 words,Entity,https://en.wikipedia.org/wiki/World_Heritage_Site


In [15]:
# 6. Resolve analysis columns in the enriched dataset
C = {
    "rid": "response_id",
    "qid": "question_id",
    "lang": "language",
    "model": "model_name",
    "response": "response",
    "question": "question",
    "gold": "ground_truth",
    "domain": find_column(df, ["domain"], required=False),
    "difficulty": find_column(df, ["difficulty"], required=False),
    "llm_label": find_column(df, ["correctness_label"], required=False),
    "llm_types": find_column(df, ["hallucination_types"], required=False),
    "llm_severity": find_column(df, ["severity"], required=False),
    "source_url": find_column(df, ["source_url"], required=False),
    "evidence": find_column(df, ["source_evidence","evidence_text","reference_evidence"], required=False)
}
display(pd.DataFrame({"logical_field": C.keys(), "actual_column": C.values()}))
print("Rows:", len(df))
print("Models:", df[C["model"]].nunique())
print("Languages:", sorted(df[C["lang"]].dropna().unique()))

,logical_field,actual_column
0,rid,response_id
1,qid,question_id
2,lang,language
3,model,model_name
4,response,response
5,question,question
6,gold,ground_truth
7,domain,domain
8,difficulty,difficulty
9,llm_label,correctness_label


Rows: 64800
Models: 6
Languages: ['EN', 'HI', 'HI-EN', 'NE', 'TA', 'TE']


In [16]:
# 7. Structural integrity and complete paired coverage
assert df[C["rid"]].notna().all() and df[C["rid"]].is_unique
assert not df.duplicated([C["qid"], C["model"], C["lang"]]).any()

models = sorted(df[C["model"]].dropna().astype(str).unique())
if len(models) != EXPECTED_MODELS:
    raise ValueError(f"Expected {EXPECTED_MODELS} models, found {len(models)}: {models}")

found_languages = set(df[C["lang"]].dropna().unique())
if found_languages != set(LANGUAGES):
    raise ValueError(f"Expected {LANGUAGES}, found {sorted(found_languages)}")

coverage = (
    df.groupby([C["lang"], C["qid"]])[C["model"]]
      .nunique().rename("n_models").reset_index()
)
bad = coverage[coverage["n_models"] != EXPECTED_MODELS]
print("Incomplete language-question groups:", len(bad))
if not bad.empty:
    display(bad.head(50))
    raise ValueError("Some questions do not have all six model responses.")

display(coverage.groupby(C["lang"]).size().rename("complete_questions").to_frame())

Incomplete language-question groups: 0


,complete_questions
language,
EN,1800
HI,1800
HI-EN,1800
NE,1800
TA,1800
TE,1800


## Sampling design

Select 100 unique question IDs per language, then retain every model response for each selected question. This produces 600 question-language units and 3,600 response records. The same questions are compared across models, preserving the paired design.

Sampling is proportionally stratified by domain and difficulty if those columns are populated. If those fields are absent or empty, it falls back to simple random sampling within each language.

In [18]:
# 8. Reproducible stratified question sampling
def stratified_question_sample(pool, n, strata_cols, seed):
    pool = pool.copy()
    strata_cols = [c for c in strata_cols if c and c in pool.columns]
    if len(pool) < n:
        raise ValueError(f"Only {len(pool)} eligible questions; need {n}.")

    usable = [c for c in strata_cols if pool[c].notna().any()]
    if not usable:
        return pool.sample(n=n, random_state=seed)

    pool["_stratum"] = pool[usable].fillna("Missing").astype(str).agg(" | ".join, axis=1)
    sizes = pool["_stratum"].value_counts().sort_index()
    raw = sizes / sizes.sum() * n
    allocation = np.floor(raw).astype(int)
    allocation = pd.Series(
        np.minimum(allocation.values, sizes.values), index=sizes.index
    )
    remainder = n - int(allocation.sum())
    order = (raw - np.floor(raw)).sort_values(ascending=False).index.tolist()

    while remainder > 0:
        changed = False
        for key in order:
            if allocation[key] < sizes[key] and remainder > 0:
                allocation[key] += 1
                remainder -= 1
                changed = True
        if not changed:
            break

    pieces = []
    for i, (key, count) in enumerate(allocation.items()):
        if count:
            pieces.append(
                pool[pool["_stratum"] == key].sample(
                    n=int(count), random_state=seed + i
                )
            )
    result = pd.concat(pieces, ignore_index=True).drop(columns="_stratum")
    if len(result) != n:
        raise RuntimeError(f"Allocation returned {len(result)} rows, expected {n}.")
    return result

metadata_cols = [C["lang"], C["qid"]]
for logical in ["domain", "difficulty"]:
    actual = C[logical]
    if actual:
        metadata_cols.append(actual)

question_meta = df[metadata_cols].drop_duplicates([C["lang"], C["qid"]])

selected_parts = []
for i, language in enumerate(LANGUAGES):
    pool = question_meta[question_meta[C["lang"]] == language].copy()
    selected = stratified_question_sample(
        pool, QUESTIONS_PER_LANGUAGE,
        [C["domain"], C["difficulty"]],
        SEED + i
    )
    selected_parts.append(selected[[C["lang"], C["qid"]]])

selected_questions = pd.concat(selected_parts, ignore_index=True)
sample = df.merge(
    selected_questions,
    on=[C["lang"], C["qid"]],
    how="inner",
    validate="many_to_one"
).sort_values([C["lang"], C["qid"], C["model"]]).reset_index(drop=True)

assert len(selected_questions) == 600
assert len(sample) == EXPECTED_RESPONSES
assert sample.groupby([C["lang"], C["qid"]])[C["model"]].nunique().eq(6).all()

display(sample.groupby([C["lang"], C["model"]]).size().unstack())
sample.to_csv(VAL / "validation_sample_internal.csv", index=False, encoding="utf-8-sig")
selected_questions.to_csv(VAL / "selected_question_ids.csv", index=False, encoding="utf-8-sig")
print("Saved internal sample:", VAL / "validation_sample_internal.csv")

model_name,Claude Sonnet 5.5,DeepSeek-V4.1-Flash,GLM-5.3 Flash,GPT-5.6 Luna,Gemini 3,Qwen 3.7
language,,,,,,
EN,100,100,100,100,100,100
HI,100,100,100,100,100,100
HI-EN,100,100,100,100,100,100
NE,100,100,100,100,100,100
TA,100,100,100,100,100,100
TE,100,100,100,100,100,100


Saved internal sample: C:\Users\CH.SC.U4CSE23160\Neural Project\IndicHallbench-Hallucinations-Benchmark\data\validation\validation_sample_internal.csv


## Blinded annotator files

Both annotators receive the same 3,600 responses, in different random orders. Their files omit Llama's correctness, hallucination, severity, confidence, and annotation-source fields.

**Before distribution:** populate and verify `source_evidence` with actual source passages. A URL alone is not sufficient evidence. If source evidence is unavailable, the annotators must be allowed to mark `InsufficientEvidence` / `Uncertain`.

In [19]:
# 9. Create blinded annotation CSVs for Annotator A and B
base = sample[[
    C["rid"], C["qid"], C["lang"], C["model"],
    C["question"], C["gold"], C["response"]
]].copy()

base = base.rename(columns={
    C["rid"]: "response_id",
    C["qid"]: "question_id",
    C["lang"]: "language",
    C["model"]: "model_name",
    C["question"]: "question",
    C["gold"]: "ground_truth",
    C["response"]: "response"
})

for logical, output_name in [
    ("domain", "domain"), ("difficulty", "difficulty"),
    ("source_url", "source_url"), ("evidence", "source_evidence")
]:
    actual = C[logical]
    base[output_name] = sample[actual].values if actual else ""

annotation_fields = [
    "correctness_label", "hallucination_types", "severity",
    "evidence_support", "claim_assessment", "reason",
    "annotator_id", "review_status"
]

for annotator, seed in [("A", SEED + 101), ("B", SEED + 202)]:
    sheet = base.sample(frac=1, random_state=seed).reset_index(drop=True)
    for field in annotation_fields:
        sheet[field] = ""
    sheet["annotator_id"] = annotator
    sheet["review_status"] = "Pending"
    output_path = ANN / f"annotator_{annotator}_blinded.csv"
    sheet.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("Created:", output_path, "| rows:", len(sheet))

print("Check source_evidence before sending the files to annotators.")

Created: C:\Users\CH.SC.U4CSE23160\Neural Project\IndicHallbench-Hallucinations-Benchmark\data\validation\annotator_files\annotator_A_blinded.csv | rows: 3600
Created: C:\Users\CH.SC.U4CSE23160\Neural Project\IndicHallbench-Hallucinations-Benchmark\data\validation\annotator_files\annotator_B_blinded.csv | rows: 3600
Check source_evidence before sending the files to annotators.


## Annotation rubric — freeze before full annotation

**Correctness**
- `Correct`: all material factual claims are supported by the verified evidence.
- `Partially Correct`: a mixture of supported and materially incorrect/unsupported claims, or a material omission under the predefined rubric.
- `Incorrect`: central answer materially false or contradicted.
- `Uncertain`: evidence insufficient or response cannot reliably be interpreted.

**Evidence support:** `Supported`, `Contradicted`, `NotEstablished`, `InsufficientEvidence`. Unsupported is not automatically false.

**Severity:** `None`, `Minor`, `Moderate`, `Critical`. Define operational examples before annotation.

**Hallucination types:** `None`, `Entity Error`, `Numerical Error`, `Temporal Error`, `Location Error`, `Fabrication`, `Contradiction`, `Other`. Separate multiple types with semicolons.

Annotators work independently, cannot see each other's labels or Llama labels, and provide a short reason. Pilot 30–50 records per language, revise ambiguous rubric rules, then freeze the rubric before the main annotation.

In [ ]:
# 10. Load and validate completed annotator files
A_PATH = ANN / "annotator_A_completed.csv"
B_PATH = ANN / "annotator_B_completed.csv"

# Copy the completed blinded files to these exact filenames after annotation.
if not A_PATH.exists() or not B_PATH.exists():
    print("Annotation is not complete yet.")
    print("Expected file A:", A_PATH)
    print("Expected file B:", B_PATH)
else:
    A = pd.read_csv(A_PATH, encoding="utf-8-sig")
    B = pd.read_csv(B_PATH, encoding="utf-8-sig")

    for name, annotator_df in [("A", A), ("B", B)]:
        required = ["response_id", "correctness_label", "hallucination_types",
                    "severity", "evidence_support", "review_status"]
        missing = [x for x in required if x not in annotator_df.columns]
        if missing:
            raise KeyError(f"Annotator {name} file missing columns: {missing}")
        if annotator_df["response_id"].duplicated().any():
            raise ValueError(f"Annotator {name} has duplicate response_id values.")
        if set(annotator_df["response_id"]) != set(sample[C["rid"]]):
            raise ValueError(f"Annotator {name} response IDs do not exactly match sample.")
        if not annotator_df["review_status"].astype(str).str.lower().eq("completed").all():
            raise ValueError(f"Annotator {name} still has pending rows.")
    print("Both annotator files passed validation.")

## Inter-annotator agreement

Calculate agreement **before adjudication**. Use nominal Cohen's Kappa for correctness, weighted Kappa for ordinal severity, and binary Kappa separately for each hallucination type. Report raw agreement and per-class confusion matrices because Kappa can be affected by prevalence imbalance. Report results overall and separately for each language.

In [ ]:
# 11. Agreement analysis
CORRECT_LABELS = ["Correct", "Partially Correct", "Incorrect", "Uncertain"]
SEVERITY_LABELS = ["None", "Minor", "Moderate", "Critical"]
HALLUCINATION_TYPES = [
    "Entity Error", "Numerical Error", "Temporal Error",
    "Location Error", "Fabrication", "Contradiction", "Other"
]

def clean_labels(series):
    return series.astype("string").str.strip().replace(
        {"": pd.NA, "nan": pd.NA, "None": pd.NA}
    )

def parse_multilabel(value):
    if pd.isna(value):
        return set()
    text = str(value).strip()
    if not text or text.lower() in {"none", "nan", "null"}:
        return set()
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return {str(x).strip() for x in parsed}
    except Exception:
        pass
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return {str(x).strip() for x in parsed}
    except Exception:
        pass
    return {x.strip() for x in re.split(r"[;|,]", text) if x.strip()}

def agreement_stats(a, b, labels=None, weights=None):
    valid = a.notna() & b.notna()
    aa, bb = a[valid].astype(str), b[valid].astype(str)
    if len(aa) == 0:
        return {"n": 0, "raw_agreement": np.nan, "kappa": np.nan}
    return {
        "n": len(aa),
        "raw_agreement": float((aa.values == bb.values).mean()),
        "kappa": float(cohen_kappa_score(aa, bb, labels=labels, weights=weights))
    }

if A_PATH.exists() and B_PATH.exists():
    AB = A.merge(B, on="response_id", suffixes=("_A", "_B"), validate="one_to_one")
    # Language field is the same in both files; verify it agrees.
    assert (AB["language_A"].astype(str) == AB["language_B"].astype(str)).all()

    rows = []
    for language, part in AB.groupby("language_A"):
        rows.append({
            "language": language,
            **agreement_stats(
                clean_labels(part["correctness_label_A"]),
                clean_labels(part["correctness_label_B"]),
                labels=CORRECT_LABELS
            )
        })
    rows.append({
        "language": "OVERALL",
        **agreement_stats(
            clean_labels(AB["correctness_label_A"]),
            clean_labels(AB["correctness_label_B"]),
            labels=CORRECT_LABELS
        )
    })
    iaa_correctness = pd.DataFrame(rows)
    display(iaa_correctness)
    iaa_correctness.to_csv(RES / "iaa_correctness.csv", index=False)

    severity_map = {label: i for i, label in enumerate(SEVERITY_LABELS)}
    sev_a = clean_labels(AB["severity_A"]).map(severity_map)
    sev_b = clean_labels(AB["severity_B"]).map(severity_map)
    print("Severity quadratic weighted Kappa:",
          cohen_kappa_score(sev_a, sev_b, weights="quadratic"))

    type_rows = []
    for label in HALLUCINATION_TYPES:
        binary_a = AB["hallucination_types_A"].apply(lambda x: label in parse_multilabel(x))
        binary_b = AB["hallucination_types_B"].apply(lambda x: label in parse_multilabel(x))
        type_rows.append({"hallucination_type": label,
                          **agreement_stats(binary_a, binary_b, labels=[False, True])})
    iaa_types = pd.DataFrame(type_rows)
    display(iaa_types)
    iaa_types.to_csv(RES / "iaa_hallucination_types.csv", index=False)
else:
    print("Complete both annotator files before running agreement analysis.")

## Adjudication

After agreement is calculated and saved, resolve disagreements. With only two available annotators, A and B may discuss disagreements **after** their independent files are locked. Report this as consensus resolution, not independent third-party adjudication. Keep the original A/B files unchanged.

Create `adjudicated_annotations.csv` with one row per response and these columns:
`response_id`, `final_correctness_label`, `final_hallucination_types`, `final_severity`, `final_evidence_support`, `adjudication_reason`, `adjudicator_id`.

If evidence cannot resolve a case, retain `Uncertain` or `InsufficientEvidence` rather than forcing agreement.

In [ ]:
# 12. Create disagreement queue
if A_PATH.exists() and B_PATH.exists():
    AB = A.merge(B, on="response_id", suffixes=("_A", "_B"), validate="one_to_one")
    AB["correctness_disagreement"] = (
        clean_labels(AB["correctness_label_A"]) != clean_labels(AB["correctness_label_B"])
    )
    AB["severity_disagreement"] = (
        clean_labels(AB["severity_A"]) != clean_labels(AB["severity_B"])
    )
    AB["evidence_disagreement"] = (
        clean_labels(AB["evidence_support_A"]) != clean_labels(AB["evidence_support_B"])
    )
    queue = AB[
        AB["correctness_disagreement"] |
        AB["severity_disagreement"] |
        AB["evidence_disagreement"]
    ].copy()
    queue.to_csv(VAL / "adjudication_queue.csv", index=False, encoding="utf-8-sig")
    print("Records requiring discussion:", len(queue))
    print("Saved:", VAL / "adjudication_queue.csv")
else:
    print("Complete both annotator files first.")

## Human–Llama judge validity

Compare the original Llama labels against adjudicated human labels for exactly the same response IDs. This estimates the Llama judge's agreement with the human consensus sample; it does not prove either source is infallible. Report macro-F1, per-class precision/recall, confusion matrices and language-specific results.

In [ ]:
# 13. Human versus Llama comparison
ADJ_PATH = VAL / "adjudicated_annotations.csv"

if ADJ_PATH.exists():
    human = pd.read_csv(ADJ_PATH, encoding="utf-8-sig")
    required_adj = ["response_id", "final_correctness_label"]
    missing_adj = [x for x in required_adj if x not in human.columns]
    if missing_adj:
        raise KeyError(f"Adjudicated file missing: {missing_adj}")
    if human["response_id"].duplicated().any():
        raise ValueError("Adjudicated response_id must be unique.")
    eval_df = sample.merge(human, on="response_id", how="inner", validate="one_to_one")
    if len(eval_df) != EXPECTED_RESPONSES:
        raise ValueError(f"Expected {EXPECTED_RESPONSES} adjudicated rows; found {len(eval_df)}.")
    if not C["llm_label"]:
        raise KeyError("The merged annotations do not contain correctness_label for Llama comparison.")

    y_true = eval_df["final_correctness_label"].astype(str)
    y_pred = eval_df[C["llm_label"]].astype(str)
    print(classification_report(
        y_true, y_pred, labels=CORRECT_LABELS, zero_division=0, digits=4
    ))

    cm = confusion_matrix(y_true, y_pred, labels=CORRECT_LABELS)
    cm_df = pd.DataFrame(cm, index=CORRECT_LABELS, columns=CORRECT_LABELS)
    display(cm_df)
    cm_df.to_csv(RES / "llama_human_confusion_matrix.csv")

    metric_rows = []
    for language, part in eval_df.groupby(C["lang"]):
        yt = part["final_correctness_label"].astype(str)
        yp = part[C["llm_label"]].astype(str)
        precision, recall, macro_f1, _ = precision_recall_fscore_support(
            yt, yp, labels=CORRECT_LABELS, average="macro", zero_division=0
        )
        metric_rows.append({
            "language": language, "n": len(part),
            "accuracy": accuracy_score(yt, yp),
            "macro_precision": precision, "macro_recall": recall,
            "macro_f1": macro_f1,
            "weighted_f1": f1_score(yt, yp, labels=CORRECT_LABELS,
                                    average="weighted", zero_division=0)
        })
    human_llm_metrics = pd.DataFrame(metric_rows)
    display(human_llm_metrics)
    human_llm_metrics.to_csv(RES / "llama_human_by_language.csv", index=False)
else:
    print("Create adjudicated_annotations.csv before this analysis.")

## Comparative model performance

Primary endpoint: **strict correctness** equals 1 only for adjudicated `Correct`; it equals 0 for `Partially Correct` and `Incorrect`. Exclude `Uncertain` from the primary denominator and report its rate separately.

Because each question is answered by all six models, comparisons are paired. Use exact McNemar tests for pairwise comparisons within each language and Holm correction for multiple comparisons. Use question-cluster bootstrap confidence intervals. Do not treat six responses to the same question as independent.

In [ ]:
# 14. Model performance and paired McNemar tests
if ADJ_PATH.exists():
    eval_df = sample.merge(
        pd.read_csv(ADJ_PATH, encoding="utf-8-sig"),
        on="response_id", how="inner", validate="one_to_one"
    )
    eval_df["strict_correct"] = (eval_df["final_correctness_label"] == "Correct").astype(int)
    eval_df["eligible_primary"] = eval_df["final_correctness_label"].isin(
        ["Correct", "Partially Correct", "Incorrect"]
    )
    eligible = eval_df[eval_df["eligible_primary"]].copy()

    model_summary = (
        eligible.groupby([C["lang"], C["model"]])["strict_correct"]
        .agg(n="count", strict_correct_rate="mean").reset_index()
    )
    model_summary["strict_correct_percent"] = 100 * model_summary["strict_correct_rate"]
    display(model_summary)
    model_summary.to_csv(RES / "model_performance_by_language.csv", index=False)

    test_rows = []
    for language, part in eligible.groupby(C["lang"]):
        wide = part.pivot(index=C["qid"], columns=C["model"], values="strict_correct")
        for model_1, model_2 in itertools.combinations(models, 2):
            paired = wide[[model_1, model_2]].dropna()
            b = int(((paired[model_1] == 1) & (paired[model_2] == 0)).sum())
            c = int(((paired[model_1] == 0) & (paired[model_2] == 1)).sum())
            p_value = 1.0 if b + c == 0 else mcnemar([[0, b], [c, 0]], exact=True).pvalue
            test_rows.append({
                "language": language, "model_1": model_1, "model_2": model_2,
                "n_paired": len(paired),
                "rate_difference_model1_minus_model2": float(paired[model_1].mean() - paired[model_2].mean()),
                "p_value": p_value
            })
    pairwise = pd.DataFrame(test_rows)
    pairwise["p_holm_within_language"] = np.nan
    for language, indices in pairwise.groupby("language").groups.items():
        pairwise.loc[indices, "p_holm_within_language"] = multipletests(
            pairwise.loc[indices, "p_value"], method="holm"
        )[1]
    display(pairwise)
    pairwise.to_csv(RES / "paired_mcnemar_holm.csv", index=False)
else:
    print("Adjudicated labels are required before comparative model analysis.")

In [ ]:
# 15. Paired question-cluster bootstrap confidence intervals
def paired_bootstrap_rates(group, question_col, model_col, outcome_col,
                           n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    question_ids = group[question_col].dropna().unique()
    model_names = sorted(group[model_col].dropna().unique())
    if len(question_ids) == 0:
        raise ValueError("No question IDs available for bootstrap.")
    by_question = {q: group[group[question_col] == q] for q in question_ids}
    bootstrap_values = {m: [] for m in model_names}

    for _ in range(n_boot):
        draw = rng.choice(question_ids, size=len(question_ids), replace=True)
        boot = pd.concat([by_question[q] for q in draw], ignore_index=True)
        rates = boot.groupby(model_col)[outcome_col].mean()
        for model in model_names:
            bootstrap_values[model].append(rates.get(model, np.nan))

    point = group.groupby(model_col)[outcome_col].mean()
    return pd.DataFrame([
        {
            "model": model,
            "estimate": point[model],
            "ci_lower": np.nanquantile(bootstrap_values[model], 0.025),
            "ci_upper": np.nanquantile(bootstrap_values[model], 0.975)
        }
        for model in model_names
    ])

if ADJ_PATH.exists():
    ci_results = []
    for language, part in eligible.groupby(C["lang"]):
        ci = paired_bootstrap_rates(
            part, C["qid"], C["model"], "strict_correct",
            n_boot=2000, seed=SEED
        )
        ci["language"] = language
        ci_results.append(ci)
    bootstrap_ci = pd.concat(ci_results, ignore_index=True)
    display(bootstrap_ci)
    bootstrap_ci.to_csv(RES / "cluster_bootstrap_model_rates.csv", index=False)

## Optional regression

A binomial GEE can test model, language and model × language interaction while clustering by question ID. If question IDs repeat across languages, create a combined language-question cluster ID. Check convergence and model assumptions. This is optional; report estimates, odds ratios, confidence intervals and p-values.

In [ ]:
# 16. Optional GEE model
if ADJ_PATH.exists():
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    gee_data = eligible.copy()
    gee_data["_question_cluster"] = (
        gee_data[C["lang"]].astype(str) + "::" +
        gee_data[C["qid"]].astype(str)
    )
    formula = f"strict_correct ~ C({C['model']}) * C({C['lang']})"
    if C["domain"]:
        formula += f" + C({C['domain']})"
    if C["difficulty"]:
        formula += f" + C({C['difficulty']})"

    gee = smf.gee(
        formula=formula,
        groups="_question_cluster",
        data=gee_data,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable()
    ).fit()
    print(gee.summary())
    (RES / "gee_model_summary.txt").write_text(
        gee.summary().as_text(), encoding="utf-8"
    )
else:
    print("Adjudicated labels are required before GEE.")

## Existing 1,015 human-reviewed records

Keep this targeted set separate. If the same person produced all 1,015 labels, they are one annotator's judgments, not inter-annotator agreement. Because the records were selected from pending/error cases, they are not representative of the full dataset. You may ask Annotator B to independently label this targeted set as a supplementary reliability exercise, but report it separately.

In [ ]:
# 17. Optional audit of the existing 1,015 records
EXISTING_1015_PATH = VAL / "targeted_error_review_1015.csv"

if EXISTING_1015_PATH.exists():
    targeted = pd.read_csv(EXISTING_1015_PATH, encoding="utf-8-sig")
    print("Targeted records:", len(targeted))
    if "response_id" in targeted.columns:
        print("Unique response IDs:", targeted["response_id"].nunique())
        print("Overlap with new validation sample:",
              targeted["response_id"].isin(sample[C["rid"]]).sum())
else:
    print("Optional file not found:", EXISTING_1015_PATH)

In [ ]:
# 18. Save reproducibility manifest and final sampling report
manifest = {
    "random_seed": SEED,
    "languages": LANGUAGES,
    "models": models,
    "questions_per_language": QUESTIONS_PER_LANGUAGE,
    "expected_validation_responses": EXPECTED_RESPONSES,
    "sampling_unit": "question_id within language; retain all six model responses",
    "annotators": 2,
    "existing_1015": "separate targeted single-reviewer error/pending subset",
    "annotation_file": str(ANNOTATION_PATH),
    "response_file": str(RESPONSE_PATH),
    "reference_file": str(REFERENCE_PATH),
    "validation_sample": str(VAL / "validation_sample_internal.csv")
}
(VAL / "validation_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

sampling_report = (
    sample.groupby([C["lang"], C["model"]])
    .size().rename("response_count").reset_index()
)
sampling_report.to_csv(RES / "validation_sample_distribution.csv", index=False)
print(json.dumps(manifest, indent=2, ensure_ascii=False))
display(sampling_report)

# Publication reporting checklist

- [ ] The three source files load successfully and their row counts are recorded.
- [ ] Annotation rows match original responses on question ID, language and model.
- [ ] Every enriched response matches one reference question.
- [ ] The six language and six model categories are verified from actual data.
- [ ] Duplicate and incomplete question-model-language groups are resolved.
- [ ] Exactly 100 question IDs per language are selected with the random seed saved.
- [ ] Actual evidence passages are supplied to annotators; URLs alone are not treated as evidence.
- [ ] Both annotators independently label the same 3,600 records while blinded to Llama labels.
- [ ] Inter-annotator agreement is calculated before adjudication.
- [ ] Original A/B labels are preserved and disagreements are resolved transparently.
- [ ] Human–Llama evaluation uses the same response IDs.
- [ ] Model comparisons preserve question pairing.
- [ ] Confidence intervals account for question-level clustering.
- [ ] Multiple comparisons are corrected.
- [ ] Existing 1,015 targeted labels are reported separately.
- [ ] Uncertain cases, class imbalance and limitations are reported.

**Power limitation:** 100 questions per language is a practical validation design, not a guarantee of adequate power for small differences or rare hallucination classes. Report uncertainty and avoid unsupported significance claims.